In [0]:
df = spark.read.csv("/Volumes/workspace/default/hr_data/HR_Dataset_dirty.csv", header=True, inferSchema=True)
df.printSchema()
print("Row count:", df.count())
df.show(5)

root
 |-- Employee_ID: integer (nullable = true)
 |-- Department: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Education_Level: string (nullable = true)
 |-- Job_Role: string (nullable = true)
 |-- Monthly_Income: double (nullable = true)
 |-- Years_At_Company: integer (nullable = true)
 |-- Years_In_Current_Role: integer (nullable = true)
 |-- Job_Satisfaction: double (nullable = true)
 |-- Performance_Rating: double (nullable = true)
 |-- Work_Life_Balance: integer (nullable = true)
 |-- Training_Hours_Last_Year: integer (nullable = true)
 |-- Last_Promotion_Years_Ago: integer (nullable = true)
 |-- Distance_From_Home: integer (nullable = true)
 |-- Overtime: string (nullable = true)
 |-- Attrition: string (nullable = true)
 |-- Marital_Status: string (nullable = true)
 |-- Number_Of_Companies_Worked: integer (nullable = true)
 |-- Stock_Option_Level: integer (nullable = true)

Row count: 3415
+-----------+----------+--------

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Null counts per column
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+-----------+----------+------+---+---------------+--------+--------------+----------------+---------------------+----------------+------------------+-----------------+------------------------+------------------------+------------------+--------+---------+--------------+--------------------------+------------------+
|Employee_ID|Department|Gender|Age|Education_Level|Job_Role|Monthly_Income|Years_At_Company|Years_In_Current_Role|Job_Satisfaction|Performance_Rating|Work_Life_Balance|Training_Hours_Last_Year|Last_Promotion_Years_Ago|Distance_From_Home|Overtime|Attrition|Marital_Status|Number_Of_Companies_Worked|Stock_Option_Level|
+-----------+----------+------+---+---------------+--------+--------------+----------------+---------------------+----------------+------------------+-----------------+------------------------+------------------------+------------------+--------+---------+--------------+--------------------------+------------------+
|          0|         0|     0|  0|           

In [0]:
# Duplicate IDs
print("Total rows:", df.count())
print("Unique Employee_IDs:", df.select("Employee_ID").distinct().count())

Total rows: 3415
Unique Employee_IDs: 3400


In [0]:
# Bad values
df.filter(col("Monthly_Income") < 0).show()
df.filter(col("Age") > 100).show()

+-----------+----------+-----------------+---+------------------+---------+--------------+----------------+---------------------+----------------+------------------+-----------------+------------------------+------------------------+------------------+--------+---------+--------------+--------------------------+------------------+
|Employee_ID|Department|           Gender|Age|   Education_Level| Job_Role|Monthly_Income|Years_At_Company|Years_In_Current_Role|Job_Satisfaction|Performance_Rating|Work_Life_Balance|Training_Hours_Last_Year|Last_Promotion_Years_Ago|Distance_From_Home|Overtime|Attrition|Marital_Status|Number_Of_Companies_Worked|Stock_Option_Level|
+-----------+----------+-----------------+---+------------------+---------+--------------+----------------+---------------------+----------------+------------------+-----------------+------------------------+------------------------+------------------+--------+---------+--------------+--------------------------+------------------+
|

In [0]:
df_clean = df.dropDuplicates(["Employee_ID"])
print("After dedup:", df_clean.count())

After dedup: 3400


In [0]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn("Monthly_Income",
    when(col("Monthly_Income") < 0, -col("Monthly_Income")).otherwise(col("Monthly_Income")))

df_clean = df_clean.withColumn("Age",
    when(col("Age") > 100, None).otherwise(col("Age")))

In [0]:
from pyspark.sql.functions import trim, initcap

df_clean = df_clean.withColumn("Department", initcap(trim(col("Department"))))

In [0]:
from pyspark.sql.functions import mean

df_clean = df_clean.na.fill({"Marital_Status": "Unknown"})

for c in ["Monthly_Income", "Performance_Rating", "Job_Satisfaction", "Age"]:
    avg_val = df_clean.select(mean(col(c))).first()[0]
    df_clean = df_clean.na.fill({c: round(avg_val, 1)})

In [0]:
df_clean = df_clean.withColumn("IncomeBand",
    when(col("Monthly_Income") < 3000, "Low")
    .when(col("Monthly_Income") < 7000, "Mid")
    .otherwise("High"))

df_clean = df_clean.withColumn("AttritionRiskFlag",
    when((col("Job_Satisfaction") <= 2) & (col("Overtime") == "Yes"), "High Risk")
    .otherwise("Low Risk"))

In [0]:
assert df_clean.filter(col("Employee_ID").isNull()).count() == 0
assert df_clean.count() == df_clean.dropDuplicates(["Employee_ID"]).count()
assert df_clean.filter(col("Monthly_Income") < 0).count() == 0
print("Validation passed. Final row count:", df_clean.count())

Validation passed. Final row count: 3400


In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("hr_employee_clean")

In [0]:
%sql
SELECT Department, ROUND(AVG(Monthly_Income),2) AS avg_income, COUNT(*) AS headcount
FROM hr_employee_clean
GROUP BY Department
ORDER BY avg_income DESC

Department,avg_income,headcount
Hr,5181.28,695
Finance,5173.57,649
It,5153.29,654
Sales,5083.51,701
Marketing,5037.71,701


In [0]:
%sql
SELECT Department, 
       ROUND(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS attrition_rate_pct
FROM hr_employee_clean
GROUP BY Department
ORDER BY attrition_rate_pct DESC

Department,attrition_rate_pct
Finance,51.2
Sales,51.2
Hr,50.8
It,50.3
Marketing,50.1


In [0]:
%sql
SELECT IncomeBand, ROUND(AVG(Job_Satisfaction),2) AS avg_satisfaction, COUNT(*) AS employees
FROM hr_employee_clean
GROUP BY IncomeBand
ORDER BY avg_satisfaction DESC

IncomeBand,avg_satisfaction,employees
High,3.11,1106
Mid,3.02,1197
Low,2.99,1097
